# NER Documentation

This notebook explores **Named Entity Recognition (NER)** for medical text using three different approaches:

1. **spaCy/scispaCy-based entity extraction**
2. **LLM-based structured information extraction**
3. **A BioBERT fine-tuning outline for token classification**

The notebook is primarily a prototype and comparison notebook. The spaCy/scispaCy section is the most complete implementation. The LLM and BioBERT sections define the extraction idea and setup, but they are not built out into a full evaluated pipeline in the current notebook.

## Objective

The goal of the notebook is to extract clinically relevant information from a medical report, such as:

- disease name
- cancer stage
- spread or metastasis information
- treatment information

The input is a medical report in plain text form, and the output depends on the approach:

- generic medical entities from a biomedical NER model
- structured JSON fields from an LLM prompt
- token-level labels in a future fine-tuned BioBERT model

## Input Text

The notebook starts with a long discharge-summary-style medical report and then replaces it with a cleaner summarized report string. That summarized text becomes the actual working input for the NER examples.

The report describes a patient with **malignant melanoma of the left foot, stage IV, with lung metastasis**, receiving immunotherapy and discharge instructions.

## Approach 1: NER Using spaCy/scispaCy

The first implemented approach uses a biomedical spaCy model:

```python
import spacy
nlp = spacy.load("en_core_sci_sm")
```

This model is a scientific/biomedical NLP pipeline and is more suitable for medical text than the default general English spaCy models.

### Extraction Logic

The notebook defines a helper function:

```python
def extract_entities(text):
    doc = nlp(text)

    entities = []
    for ent in doc.ents:
        entities.append((ent.text, ent.label_))

    return entities
```

This function:

- processes the medical report with the biomedical NLP model
- extracts all recognized named entities
- returns them as `(entity_text, entity_label)` tuples

The notebook then also prints each entity with character offsets:

```python
for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)
```

This is useful for debugging and understanding exactly where the model found each entity in the report.

### Visualization

The spaCy section uses `displacy` to render the detected entities:

```python
from spacy import displacy
displacy.render(doc, style='ent')
```

This gives a visual view of the extracted spans in the text.

### What This Approach Provides

The spaCy/scispaCy pipeline provides **pretrained biomedical NER**, which is useful when:

- quick extraction is needed
- no labeled training dataset is available
- the goal is exploratory analysis of medical text

However, this approach returns the entities recognized by the pretrained model and does not directly guarantee the exact custom schema needed by the project, such as only `disease`, `stage`, `spread`, and `treatment`.

## Approach 2: NER Using an LLM

The notebook then defines a structured extraction approach using a large language model.

### Target Output Schema

The intended structured output is:

```json
{
  "disease": "",
  "stage": "",
  "spread": "",
  "treatment": ""
}
```

This moves the task from open-ended entity detection to **schema-guided information extraction**.

### Prompt Design

The notebook builds a prompt that instructs the model to return only JSON:

```python
def build_ner_prompt(report_text):
    return f"""
You are a medical information extraction system.

Extract the following fields from the medical report:
- disease
- stage
- spread
- treatment

Rules:
- Return ONLY valid JSON
- Do not add explanations
- If a field is missing, return null
- Keep answers concise

Medical Report:
{report_text}
"""
```

This is a standard prompt-engineering pattern for structured extraction from free text.

### Model Choice

The notebook loads:

```python
model_name = "BioMistral/BioMistral-7B"
```

and uses Hugging Face `AutoTokenizer` and `AutoModelForCausalLM` to generate a response.

### JSON Parsing

The notebook includes a helper to extract the JSON block from the model response:

```python
def parse_json(response):
    match = re.search(r"\{.*\}", response, re.DOTALL)
    if match:
        return json.loads(match.group())
    return None
```

This is useful because generative models often return extra text around the desired JSON output.

### What This Approach Provides

The LLM approach is intended to directly extract the exact fields required by the application. Compared with standard NER, it is more flexible for custom schemas and can combine multiple mentions into one structured answer.

In the current notebook, this section defines the setup and extraction functions, but it does not include saved outputs, evaluation, or production safeguards.

## Approach 3: Fine-Tuning BioBERT for NER

The final section sketches a more formal supervised NER pipeline using BioBERT.

### Labeling Format

The notebook shows an example BIO tagging scheme:

```text
Invasive     B-DISEASE
ductal       I-DISEASE
carcinoma    I-DISEASE
Stage        B-STAGE
II           I-STAGE
lymph        B-SPREAD
node         I-SPREAD
```

This indicates the intended direction: token-level supervised training with custom entity classes.

### Model Setup

The notebook references the BioBERT checkpoint:

```python
model = "dmis-lab/biobert-base-cased-v1.1"
NUM_LABELS = 10
```

and initializes `AutoModelForTokenClassification`.

This would be the correct general direction for training a domain-specific NER model, but the notebook currently does not include:

- a labeled training dataset
- tokenization and label alignment
- training arguments
- a trainer loop
- evaluation metrics

So this section is best understood as a **starting outline**, not a finished fine-tuning pipeline.

## End-to-End Summary of the Notebook

The notebook presents three levels of NER maturity:

- **Pretrained biomedical NER** with spaCy/scispaCy for immediate entity extraction
- **Prompt-based structured extraction** with an LLM for JSON output
- **Supervised BioBERT fine-tuning plan** for a future domain-specific token classifier

## Strengths

- explores multiple NER strategies in one notebook
- uses a biomedical NLP model rather than a generic English model
- includes both entity extraction and schema-based extraction ideas
- shows the direction toward a trainable domain-specific NER system

## Current Limitations

- no evaluation metrics are included
- no saved outputs are recorded for the spaCy or LLM sections
- the LLM section is a prototype and may return inconsistent JSON in practice
- the BioBERT section is incomplete and not yet trainable as written
- the notebook uses one main report example rather than a broader benchmark set

## Summary

This notebook is a medical NER exploration notebook. It starts with a biomedical pretrained entity recognizer, then moves to LLM-based structured extraction, and finally sketches how a custom BioBERT NER model could be fine-tuned. The strongest implemented part is the spaCy/scispaCy entity extraction pipeline, while the LLM and BioBERT sections define the next steps toward more task-specific medical information extraction.

In [ ]:
medical_report_text = """

tient Name
:Mrs. MALLAVARAPU MANOHARAMMA
ESTIGATION
Parameters
te:
RSE IN THE HOSPITAL
SYSTEMIC EXAMINATION:
Omega
Admission No: IPT1437
Hospitals
CVS-S1, S2, heard, No murmurs.
RS-Bilateral AE present.
P/A-Soft, No organomegaly
CNS-No focal neurological deficits.
: Enclosed.
Result Value
Normal Value
: Patient is a K/C/O Malignant melanoma of left foot / stage iv with lung metastasis. Admitted for I cycle immunotherapy. Routine blood investigations were done and found to be within normal limits. After necessary premedications, patient was treated with
Inj. NIVOLUMAB
Inj. IPILIMUMAB
140mg
iv infusion
50mg
iv infusion
iv infusion
Inj. ZOLEDRONIC ACID 4mg
Post chemotherapy uneventful, patient was stable and discharged with the following advice.
ION ON DISCHARGED : Stable.
RGE ADVICE
VE PLAN OF CARE
GEMENT BY TENDANT
ti
: Rx
Hospitals
1. Tab. CALCEFERON PLUS
2. Tab. LEVOCETIRIZINE
AF 0-1-0X21days
SOS X10
3. Continue HTN medications as adviced.
:- Avoid uncooked / outside food
Wear a face mask always
- Drink 3 liters of water/ fluids per day.
:- Review on 26/02/2026 with CBP, LFT, RFT, serum calcium.
- Review SOS incase of fever, vomitings, loose stools, pain abdomen. Emergency contact-8977730062
: The discharge team explained discharge instructions in the language I understand. I have been given the opportunity to ask questions concerning the discharge instructions and my questions have been answered to my satisfaction. I was handed over a copy of discharge summary at the time of discharge.
5-141, C Mallavaram, Opp. Gandhi Nagar,
or Bypass Road, Opp. Vakulamata Devi Temple,
Ci-517505
0877 3501609/91003 39991
info@omegahospitals.com
www.omegahospitals.com
COCUS-nr
"""

In [ ]:
medical_report_text = "The medical report indicates that a patient who is a known case of malignant melanoma of the left foot, stage IV with lung metastasis, was admitted to a hospital for immunotherapy. The patient's routine blood tests were normal, and the patient was then treated with nitro mustard and ipilimumab. After the patient stabilized, they were discharged with instructions to take calcium supplements and continue their high blood pressure medications. The plan of care included avoiding uncooked or outside food, wearing a face mask at all times, and drinking three liters of fluids per day. The patient was to be reviewed two weeks later with blood pressure, liver function tests, renal function tests, serum calcium, in case of fever, vomiting, loose stools, or pain in the abdomen, the patient was to present to the hospital. The patient was also given an emergency contact number to use in case of an emergency. The patient stated that they understood the discharge instructions and had their questions answered to their satisfaction. A copy of the discharge summary was also given to the patient. The hospital's contact information was also provided."

## NER using Spacy

In [ ]:
import spacy

nlp = spacy.load("en_core_sci_sm")

In [ ]:
def extract_entities(text):
    doc = nlp(text)

    entities = []
    for ent in doc.ents:
        entities.append((ent.text, ent.label_))

    return entities

In [ ]:
data = extract_entities(medical_report_text)

print(data)

In [ ]:
doc = nlp(medical_report_text)
for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

In [ ]:
from spacy import displacy

displacy.render(doc, style='ent')

## NER using LLM

In [ ]:
# Defining output Schema

{
  "disease": "",
  "stage": "",
  "spread": "",
  "treatment": ""
}

In [ ]:
# Prompt Design

def build_ner_prompt(report_text):
    return f"""
You are a medical information extraction system.

Extract the following fields from the medical report:
- disease
- stage
- spread
- treatment

Rules:
- Return ONLY valid JSON
- Do not add explanations
- If a field is missing, return null
- Keep answers concise

Medical Report:
{report_text}
"""

In [ ]:
# using biomistral for NER

from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "BioMistral/BioMistral-7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def extract_patient_data(report_text):
    prompt = build_ner_prompt(report_text)

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=200)
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response

In [ ]:
# Exrtacting JSON

import json
import re

def parse_json(response):
    match = re.search(r"\{.*\}", response, re.DOTALL)
    if match:
        return json.loads(match.group())
    return None

## Fine Tuning an BioBERT model for NER

### Prepare Dataset

In [ ]:
#example 
'''
Invasive     B-DISEASE
ductal       I-DISEASE
carcinoma    I-DISEASE
Stage        B-STAGE
II           I-STAGE
lymph        B-SPREAD
node         I-SPREAD
'''

### Load Model

In [ ]:
model = "dmis-lab/biobert-base-cased-v1.1"
NUM_LABELS = 10

from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=NUM_LABELS)
